<a href="https://colab.research.google.com/github/baddy2002/image-gs/blob/main/GaussianSplatting3D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Clona il repository

!git clone https://github.com/baddy2002/image-gs

%cd image-gs

In [ ]:
# 3. Installazione dipendenze base
!pip install ninja lpips pytorch-msssim

# 4. Installiamo il gsplat locale
%cd gsplat
!pip install .
%cd ..

In [ ]:
import os

model_path = "/content/image-gs/model.py"

with open(model_path, "r") as f:
    lines = f.readlines()

with open(model_path, "w") as f:
    for line in lines:
        # Sostituiamo l'import che dava errore
        if "from fused_ssim import fused_ssim" in line:
            f.write("from pytorch_msssim import ssim as fused_ssim\n")
        # Correggiamo la chiamata alla funzione (pytorch-msssim vuole data_range)
        elif "ssim_val = fused_ssim(images.unsqueeze(0), self.gt_images.unsqueeze(0))" in line:
            f.write("            ssim_val = fused_ssim(images.permute(2,0,1).unsqueeze(0), self.gt_images.permute(2,0,1).unsqueeze(0), data_range=1.0)\n")
        else:
            f.write(line)

print("✅ model.py modificato! Ora usa la SSIM standard e non darà più errori di import.")

In [ ]:
import os

path = "/content/image-gs/utils/image_utils.py"

# Leggiamo il file originale
with open(path, "r") as f:
    lines = f.readlines()

with open(path, "w") as f:
    for line in lines:
        # Disattiviamo LaTeX che spesso dà errori su Colab se non installato
        if "plt.rcParams['text.usetex'] = True" in line:
            f.write("plt.rcParams['text.usetex'] = False\n")
        # Ripariamo la riga della discordia in fondo al file
        elif "flip_error_map, _, _ =" in line:
            f.write("        # flip_error_map rimosso per compatibilità\n")
            f.write("        continue\n")
        # Commentiamo gli import problematici
        elif "import flip_evaluator" in line:
            f.write("# import flip_evaluator\n")
        else:
            f.write(line)

print("✅ image_utils.py è stato riparato correttamente!")
!python main.py --help

In [ ]:

# 6. Test finale
!python main.py --help

In [34]:
# Installa lo strumento di conversione
!apt-get install imagemagick -y

# Ridimensiona il vaso a 1024px (molto più gestibile per una T4)
!convert "media/textures/castpol01.png" -resize 1024x1024 "media/textures/castpol01_small.png"

# Verifica la nuova dimensione (dovrebbe essere pochi KB/MB invece di 32MB)
!ls -lh media/textures/castpol01_small.png

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
imagemagick is already the newest version (8:6.9.11.60+dfsg-1.3ubuntu0.22.04.5).
0 upgraded, 0 newly installed, 0 to remove and 45 not upgraded.
-rw-r--r-- 1 root root 935K Mar 27 21:09 media/textures/castpol01_small.png


Lancia con quantizzazione

In [35]:
!python main.py --input_path="textures/castpol01_small.png" --exp_name="test/castpol01_small" --num_gaussians=10000 --quantize

[2026/03/27 21:10:31] Start optimizing 10000 Gaussians for 'textures/castpol01_small.png'
[2026/03/27 21:10:31] ***********************************************
[2026/03/27 21:10:31] Uncompressed: 3145.73 KB | 24.000 bpp | 8.000 bppc
[2026/03/27 21:10:31] Compressed: 160.00 KB | 1.221 bpp | 0.407 bppc
[2026/03/27 21:10:31] Compression rate: 19.66x | 5.09%
[2026/03/27 21:10:31] ***********************************************
[2026/03/27 21:10:31] Image gradient map successfully saved
[2026/03/27 21:10:31] ***********************************************
[2026/03/27 21:10:35] Step: 100 | Total: 2.76 s | Render: 0.28 s | Loss: 0.0185, L1: 0.0185, SSIM: 0.0000 | PSNR: 26.14 | SSIM: 0.9999
[2026/03/27 21:10:38] Step: 200 | Total: 5.47 s | Render: 0.55 s | Loss: 0.0146, L1: 0.0146, SSIM: 0.0000 | PSNR: 27.75 | SSIM: 0.9999
[2026/03/27 21:10:41] Step: 300 | Total: 8.22 s | Render: 0.83 s | Loss: 0.0131, L1: 0.0131, SSIM: 0.0000 | PSNR: 28.63 | SSIM: 0.9999
[2026/03/27 21:10:44] Step: 400 | Tota

In [36]:
!python main.py --input_path="textures/castpol01.png" --exp_name="test/castpol01" --num_gaussians=10000 --quantize --eval --render_height=3000

[2026/03/27 21:14:25] Start rendering 10000 Gaussians for 'textures/castpol01.png'
[2026/03/27 21:14:25] ***********************************************
[2026/03/27 21:14:29] Uncompressed: 201326.59 KB | 24.000 bpp | 8.000 bppc
[2026/03/27 21:14:29] Compressed: 160.00 KB | 0.019 bpp | 0.006 bppc
[2026/03/27 21:14:29] Compression rate: 1258.29x | 0.08%
[2026/03/27 21:14:29] ***********************************************
Traceback (most recent call last):
  File "/content/image-gs/main.py", line 56, in <module>
    main(arguments)
  File "/content/image-gs/main.py", line 44, in main
    ImageGS = GaussianSplatting2D(args)
              ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/image-gs/model.py", line 60, in __init__
    self._load_model()
  File "/content/image-gs/model.py", line 278, in _load_model
    raise FileNotFoundError(f"No checkpoint found in '{self.ckpt_dir}'")
FileNotFoundError: No checkpoint found in 'results/test/castpol01/num-10000_inv-scale-5.0_bits-16-16-16-16_top-10_g